# Integration of Vector DB Context Pipeline with LLM output
- Notebook by Adam Lang
- Date: 8-17-2026
- In this notebook we will implement a simple RAG pipeline using the Vector DB.

In [8]:
from langchain_groq import ChatGroq
### document data structure
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
import os
from dotenv import load_dotenv
load_dotenv()

## init groq LLM (set API key in environment)
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

C:\Users\pytho\AppData\Local\Temp\ipykernel_8584\2598656740.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


## Initiate LLM from GROQ API

In [24]:
## init groq llm API
from langchain.chat_models import init_chat_model

# llm=init_chat_model("groq:llama-3.3-70b-versatile") ## model being phased out on Aug 16, 2026
llm = init_chat_model("groq:openai/gpt-oss-120b",
                      temperature=0.1, max_tokens=1024) ## new model
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.5', 'langchain': '1.3.15'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000022F30703850>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000022F3070E650>, model_name='openai/gpt-oss-120b', temperature=0.1, model_kwargs={}, groq_api_key=SecretStr('**********'), max_tokens=1024)

## Embedding and Vector DB
- Now we will demo how to create embeddings of documents and store them in a Vector DB.
- Quick note about embeddings and why using `numpy.ndarray`:
    Using numpy.ndarray for sentence embeddings is generally preferred over PyTorch tensors for post-processing and storage because NumPy arrays integrate natively with standard data science libraries, require no gradient tracking overhead, and consume less memory for non-training tasks.

In [10]:
### imports
import numpy as np
from sentence_transformers import SentenceTransformer ## embeddings
import chromadb ## vector db
from chromadb.config import Settings
import uuid ## record ids inserted into vector DB
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

### Create Embeddings

In [11]:
## create embedding class
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize embedding manager
        
        Args:
            model_name: HuggingFace model name for SentenceTransformers
        
        """
        self.model_name = model_name
        self.model = None ## later on init value
        self._load_model()

    ## load model function
    def _load_model(self):
        """Load SentenceTransformer model"""
        try:
            print("Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    ## generate embeddings function --> numpy array
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)

        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generate embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## init embeddings manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: {self.model_name}


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 958.97it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\pytho\AppData\Local\Temp\ipykernel_8584\1780747529.py:23: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### Create Vector Store

In [12]:
### vector store class
class VectorStore:
    """Manages document embeddings in ChromaDB vector store."""


    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of ChromaDB collection
            persist_directory: Directory to persis the vector store
        
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()


    ## init vector store
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True) # create new if doesn't already exist
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            # NOTE: hnsw:space is only honored at collection *creation* time -- Chroma
            # defaults to squared L2 distance otherwise. We embed with all-MiniLM-L6-v2
            # and want distance == cosine distance, so set it explicitly.
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG", "hnsw:space": "cosine"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")


        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    ## add documents to collection
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = [] ## uuids
        metadatas = []
        documents_text = []
        embeddings_list = []

        ## loop to embed documents and insert 
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}" ## uuid for specific record inserted
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['context_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # document content
            documents_text.append(doc.page_content)

            # embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text,
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


## init vector store
vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 1750


In [13]:
## ONE-TIME MIGRATION: this collection was originally created without hnsw:space,
## so Chroma built its HNSW index using squared L2 distance. hnsw:space can only be
## set at creation time, so to actually get cosine distance we have to recreate the
## collection and re-add the existing records (embeddings are reused, not recomputed).
## Safe to re-run: it's a no-op once the collection already reports "cosine".
_current_space = vectorstore.collection.metadata.get("hnsw:space") if vectorstore.collection.metadata else None

if _current_space != "cosine":
    print(f"Collection space is currently {_current_space!r}. Migrating to cosine...")
    _existing = vectorstore.collection.get(include=["embeddings", "documents", "metadatas"])
    _count = len(_existing["ids"])

    vectorstore.client.delete_collection(name=vectorstore.collection_name)

    vectorstore.collection = vectorstore.client.get_or_create_collection(
        name=vectorstore.collection_name,
        metadata={"description": "PDF document embeddings for RAG", "hnsw:space": "cosine"},
    )

    if _count:
        vectorstore.collection.add(
            ids=_existing["ids"],
            embeddings=_existing["embeddings"],
            metadatas=_existing["metadatas"],
            documents=_existing["documents"],
        )
    print(f"Migrated {_count} records. New space: {vectorstore.collection.metadata.get('hnsw:space')}")
else:
    print("Collection already uses cosine space -- no migration needed.")

Collection already uses cosine space -- no migration needed.


## Text Splitting (Chunking)

In [14]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [15]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: the-history-of-india-john-mcleod.pdf
  ✓ Loaded 281 pages

Total documents loaded: 281


In [16]:
## print all pdfs
all_pdf_documents

[Document(metadata={'producer': 'Adobe Acrobat 8.1; modified using iTextSharp™ 5.5.2 ©2000-2014 iText Group NV (AGPL-version)', 'creator': 'Arbortext Advanced Print Publisher 9.1.405/W Unicode', 'creationdate': '2014-12-31T19:40:04+05:30', 'author': 'John McLeod', 'ebx_publisher': 'ABC-CLIO', 'keywords': '', 'moddate': '2015-07-19T21:45:00+01:00', 'title': 'The History of India', 'source': '..\\data\\pdf\\the-history-of-india-john-mcleod.pdf', 'total_pages': 281, 'page': 0, 'page_label': 'Cover', 'source_file': 'the-history-of-india-john-mcleod.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'Adobe Acrobat 8.1; modified using iTextSharp™ 5.5.2 ©2000-2014 iText Group NV (AGPL-version)', 'creator': 'Arbortext Advanced Print Publisher 9.1.405/W Unicode', 'creationdate': '2014-12-31T19:40:04+05:30', 'author': 'John McLeod', 'ebx_publisher': 'ABC-CLIO', 'keywords': '', 'moddate': '2015-07-19T21:45:00+01:00', 'title': 'The History of India', 'source': '..\\data\\

In [17]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [18]:
chunks=split_documents(all_pdf_documents)
chunks

Split 281 documents into 875 chunks

Example chunk:
Content: THE HISTORY
OF INDIA...
Metadata: {'producer': 'Adobe Acrobat 8.1; modified using iTextSharp™ 5.5.2 ©2000-2014 iText Group NV (AGPL-version)', 'creator': 'Arbortext Advanced Print Publisher 9.1.405/W Unicode', 'creationdate': '2014-12-31T19:40:04+05:30', 'author': 'John McLeod', 'ebx_publisher': 'ABC-CLIO', 'keywords': '', 'moddate': '2015-07-19T21:45:00+01:00', 'title': 'The History of India', 'source': '..\\data\\pdf\\the-history-of-india-john-mcleod.pdf', 'total_pages': 281, 'page': 1, 'page_label': 'i', 'source_file': 'the-history-of-india-john-mcleod.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Adobe Acrobat 8.1; modified using iTextSharp™ 5.5.2 ©2000-2014 iText Group NV (AGPL-version)', 'creator': 'Arbortext Advanced Print Publisher 9.1.405/W Unicode', 'creationdate': '2014-12-31T19:40:04+05:30', 'author': 'John McLeod', 'ebx_publisher': 'ABC-CLIO', 'keywords': '', 'moddate': '2015-07-19T21:45:00+01:00', 'title': 'The History of India', 'source': '..\\data\\pdf\\the-history-of-india-john-mcleod.pdf', 'total_pages': 281, 'page': 1, 'page_label': 'i', 'source_file': 'the-history-of-india-john-mcleod.pdf', 'file_type': 'pdf'}, page_content='THE HISTORY\nOF INDIA'),
 Document(metadata={'producer': 'Adobe Acrobat 8.1; modified using iTextSharp™ 5.5.2 ©2000-2014 iText Group NV (AGPL-version)', 'creator': 'Arbortext Advanced Print Publisher 9.1.405/W Unicode', 'creationdate': '2014-12-31T19:40:04+05:30', 'author': 'John McLeod', 'ebx_publisher': 'ABC-CLIO', 'keywords': '', 'moddate': '2015-07-19T21:45:00+01:00', 'title': 'The History of India', 'sou

## Convert Chunks into Embeddings

In [19]:
## convert chunks into embeddings
texts=[doc.page_content for doc in chunks]


## Generate embeddings
embeddings=embedding_manager.generate_embeddings(texts)

## store in vector DB -- chunks + embeddings
vectorstore.add_documents(chunks, embeddings)

Generate embeddings for 875 texts...


Batches: 100%|██████████| 28/28 [02:51<00:00,  6.13s/it]


Generated embeddings with shape: (875, 384)
Adding 875 documents to vector store...
Successfully added 875 documents to vector store
Total documents in collection: 2625


## RAG Retriever

In [20]:
## RAG retriever pipeline
class RAGRetriever:
    """Handles query-based retrieval from vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    ## retrieval function
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float=0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for an incoming user query.

        Args:
            query: The user search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
        
        Returns:
            List of dictionaries containing retrieved documents and metadata
        
        
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )

            ## Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
        
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Collection is configured with hnsw:space="cosine", so Chroma's
                    # distance is already cosine distance (1 - cos_sim).
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1,

                        })
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found.")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


## create rag retriever
rag_retriever=RAGRetriever(vectorstore, embedding_manager)
rag_retriever

In [21]:
## call retriever --> get context
rag_retriever.retrieve("Who is Ghandi?")

Retrieving documents for query: 'Who is Ghandi?'
Top K: 5, Score threshold: 0.0
Generate embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 14.18it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_9968e694_852',
  'content': 'Gaharwars, 40–41\nGandhi, Indira, 4, 158–77\nGandhi, Mohandas Karamchand\n(Mahatma), 115–20,\n123–41, 184\nGandhi, Rahul, 206, 211–12\nGandhi, Rajiv , 172, 177–88, 197\nGandhi, Sanjay , 169–70, 172\nGandhi, Sonia, 5, 188, 200–202,\n206, 211\nGanga Singh, maharaja of\nBikaner, 113\nGanges, 2, 19, 29\nGautama. See Buddha\nGeography , 1–3\nGhauri, Muizz ud-Din\nMuhammad, 40–41\nGhauris, 40–41\nGhaznawids, 40\nGhiyasids, 42\nGhulam Ahmad, Mirza. See\nAhmad, Mirza Ghulam\nGobind Singh (Gobind Das), 64,\n67, 102–3\nGokhale, Gopal Krishna, 108–9,\n113, 118, 129\nGolkonda sultanate, 53–54,\n57, 62, 66\nGovernment: ancient and\nmedieval, 16–20, 23–27,\n36–38, 40, 46–49, 53–54;\ncolonial, 77–79, 83–84, 93–95,\n105–9; Montagu-Chelmsford\nreforms, 113–15; Morley-\nMinto reforms, 109–10;\nMughal, 57–60. See also\nConstitution of 1950; Govern-\nment of India Act (1935)\nGovernment of India Act (1935),\n123–31\nGowda, Haradanahalli\nDoddegowda Deve. See Deve\

In [22]:
## simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    """Retrieve the context"""
    results=retriever.retrieve(query, top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the query."

    ## generate answer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}


        Answer:"""

    ## create response
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [25]:
## now we can call the function
answer=rag_simple("What is the capital of India?", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'What is the capital of India?'
Top K: 3, Score threshold: 0.0
Generate embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.61it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


New Delhi.


## Enhanced RAG Pipeline

In [30]:
## lets enhance the RAG pipeline
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with enhanced features:
    - Return answer, scores, confidence score, and optionally the full context. 
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''} # context from Vector DB

    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results]) ## combine results
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:500] + '...' ## display up to 500 chars
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    ## Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer: """
    response = llm.invoke([prompt.format(context=context, query=query)])

    ## get output
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence,
    }
    if return_context:
        output['context'] = context
    return output

## Example usage:
result = rag_advanced("What is the Mughal dynasty?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])


Retrieving documents for query: 'What is the Mughal dynasty?'
Top K: 3, Score threshold: 0.1
Generate embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.70it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Answer: The Mughal dynasty was a line of Muslim emperors who ruled most of the Indian subcontinent from the early 16th to the early 18th centuries. Founded by Babur in 1526 after his conquest of the Delhi Sultanate, the dynasty included notable rulers such as Akbar, Shah Jahan, and Aurangzeb, and is known for its extensive territorial expansion, centralized administration, and cultural achievements.
Sources: [{'source': 'the-history-of-india-john-mcleod.pdf', 'page': 236, 'score': 0.6911121010780334, 'preview': 'Ashoka (?–c. 235 BCE). Ruler. The second king of Magadha of\nthe Mauryan dynasty (c. 272–235 BCE); ruled much of modern India;\nthe ﬁrst known king in South Asian to set up inscriptions; became a\nBuddhist and adopted the ethical policy of dhamma; considered one\nof the greatest rulers of ancient India.\nAurangzeb (1618–1707). Ruler. The sixth emperor of the Mughal\ndynasty (1658–1707) and son and

## Advanced RAG Pipeline -- Streaming, Citations, History, Summarization

In [32]:
## further RAG enhancements
from typing import List, Dict, Any
import time 


## class for enhanced pipeline
class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        ## Add citations to answer from RAG pipeline
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("Why did the British come to India?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'Why did the British come to India?'
Top K: 3, Score threshold: 0.1
Generate embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 11.52it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
now called the Indian Army , included half of all the troops in the
British Empire in the l

ate nineteenth century . It was paid for by Indian
taxpayers, as were the salaries of Britons who found jobs in the
administration and military in India. Up to the 1930s, more than one
fourth of all the taxes collected in India ﬂowed into Britain in the form
of “Home Charges,” which paid for military supplies, pensions for
British retirees from Indian services, the expenses of the India Ofﬁce,
and interest on money that the Ind ian administration had borrowed
in London. India was the world’s largest market for British exports,
and supplied raw materials, foodstuffs, and manufactured goods in
return. Indian laborers worked for minimal wages on British planta-
tions around the Indian Ocean and the Caribbean, and on railroad
construction in British East Africa.
Nevertheless, the British did not want India simply for economic
reasons. The subcontinent would probably have imported just as

now called the Indian Army , included half of all the troops in the
British Empire in the late ninetee